# 04B 真实 CIF → descriptors → ML table

> 🔵 **Level B · 建议掌握** | 科研实践 | 完成标准：建立可复现的 `CIF → descriptor row → COF_ID → target row` 映射。

向前追溯一步：`real COF CIF → parse → QC → descriptors → COF_ID → target table → ML`。使用 CURATED-COFs 的真实结构。


## 本实验的 target 与边界
密度来自真实 CIF，但这里训练密度模型是数据管线练习，不是吸附预测。完整结构的质量/体积可以直接算密度；排除原子数和体积也不代表消除了关联，因为晶格参数仍编码体积信息。科研预测应连接独立来源的 target。


## 1. 先看完整科研数据流

```text
CIF files
   ↓
pymatgen Structure
   ↓
composition + lattice + simple geometry descriptors
   ↓
feature table
   ↓
COF_ID
   ↓
target table
   ↓
merge / quality control
   ↓
train / test
   ↓
baseline model
   ↓
evaluation + interpretation
```

其中最容易被初学者忽略的是 **COF_ID**。结构、描述符、模拟结果和实验性质通常来自不同文件；没有稳定 ID，就无法可靠地把同一个材料的数据连接起来。


In [ ]:
!pip -q install pymatgen scikit-learn
import io
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pymatgen.core import Structure
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. 读取真实 COF 清单

CURATED-COFs 提供 COF ID、名称、元素组成和修正记录，并在 `cifs/` 中提供对应 CIF。

教学时先抽取一小批结构。真实项目可以把 `N_COF` 调大，但应记录下载失败和解析失败的结构，而不是静默丢弃。


In [ ]:
META_URL = "https://raw.githubusercontent.com/Wanteen/CURATED-COFs/master/cof-frameworks.csv"
meta = pd.read_csv(META_URL)
meta = meta.rename(columns={"CURATED-COFs ID": "COF_ID"})
display(meta.head())
print("database rows =", len(meta))

N_COF = 80
subset = meta.head(N_COF).copy()


## 3. CIF → Structure → descriptors

下面的函数只提取**可以直接从 CIF / pymatgen 得到**的基础描述符：

- 晶格：`a, b, c, α, β, γ`
- 晶胞体积
- 原子数
- 元素数
- H/C/N/O/B/F/S 等元素原子分数
- 一个简单的晶胞各向异性指标

注意：**LCD、PLD、ASA、void fraction、pore volume 并不能仅靠这段 pymatgen 代码可靠得到。** 这些孔结构描述符通常需要 Zeo++、PoreBlazer 或专门的孔隙分析工作流。不要把“能从 CIF 读取”与“能直接从 pymatgen 算出所有孔性质”混为一谈。


In [ ]:
ELEMENTS = ["H","B","C","N","O","F","S"]

def structure_record(cof_id, structure):
    comp = structure.composition
    n = len(structure)
    rec = {
        "COF_ID": cof_id,
        "a_A": structure.lattice.a,
        "b_A": structure.lattice.b,
        "c_A": structure.lattice.c,
        "alpha_deg": structure.lattice.alpha,
        "beta_deg": structure.lattice.beta,
        "gamma_deg": structure.lattice.gamma,
        "cell_volume_A3": structure.volume,
        "n_atoms": n,
        "n_elements": len(comp.elements),
        "density_g_cm3": float(structure.density),
    }
    abc = np.array([rec["a_A"], rec["b_A"], rec["c_A"]], dtype=float)
    rec["cell_anisotropy"] = abc.max() / abc.min()
    for el in ELEMENTS:
        rec[f"frac_{el}"] = comp.get_atomic_fraction(el)
    return rec

records, failed = [], []
for cof_id in subset["COF_ID"]:
    url = f"https://raw.githubusercontent.com/Wanteen/CURATED-COFs/master/cifs/{cof_id}.cif"
    try:
        r = requests.get(url, timeout=20)
        r.raise_for_status()
        s = Structure.from_str(r.text, fmt="cif")
        records.append(structure_record(cof_id, s))
    except Exception as exc:
        failed.append((cof_id, type(exc).__name__ + ": " + str(exc)))

features_all = pd.DataFrame(records)
print("parsed =", len(features_all), "failed =", len(failed))
display(features_all.head())
if failed:
    print("first failures:", failed[:10])

if len(features_all) < 10:
    raise RuntimeError(f"Too few parsed structures: {len(features_all)}; inspect failures: {failed[:5]}")
display(pd.DataFrame(failed, columns=["COF_ID", "error"]).head(10))


## 4. 先做结构质量检查，而不是立刻训练

真实 CIF 常见问题包括 guest/solvent、缺 H、disorder、partial occupancy、非标准晶胞、重复结构，以及不合理的层间堆积。

这里先用最基础的统计找异常值。发现异常后，科研项目中应该回到 CIF 和数据来源检查，而不是只用 `dropna()` 把问题隐藏掉。


In [ ]:
qc_cols = ["a_A","b_A","c_A","cell_volume_A3","n_atoms","density_g_cm3","cell_anisotropy"]
display(features_all[qc_cols].describe().T)

print("missing values:")
display(features_all.isna().sum().sort_values(ascending=False).head(10))


## 5. 用 COF_ID 把 feature 和 target 重新连接

为了明确练习“不同数据源如何映射”，我们故意把刚才的表拆成两部分：

- `feature_table`：模型输入
- `target_table`：目标性质

然后只通过 `COF_ID` 合并。

在真正的 CO₂ 吸附项目中，`target_table` 可能来自 GCMC、实验数据库或论文整理；关键操作仍然是同一个 `merge()`。


In [ ]:
target_table = features_all[["COF_ID","density_g_cm3"]].copy()

feature_cols = [
    "a_A","b_A","c_A","alpha_deg","beta_deg","gamma_deg",
    "cell_anisotropy","n_elements",
    "frac_H","frac_B","frac_C","frac_N","frac_O","frac_F","frac_S"
]
feature_table = features_all[["COF_ID"] + feature_cols].copy()

dataset = feature_table.merge(target_table, on="COF_ID", how="inner", validate="one_to_one")
print("feature rows =", len(feature_table))
print("target rows  =", len(target_table))
print("merged rows  =", len(dataset))
display(dataset.head())


### 排除直接组成项仍不代表“无捷径”
去掉体积和原子数只是构造较弱的教学基线，晶格参数仍决定体积。先与质量/体积的直接计算比较，再解释模型分数；不要把这一练习当成发现了独立的结构—吸附规律。


## 6. 训练 baseline

真实科研中需要更严格的结构去重、family/topology-aware split、交叉验证和外部测试集。本章只完成最基本的闭环。


In [ ]:
X = dataset[feature_cols]
y = dataset["density_g_cm3"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("MAE  =", mean_absolute_error(y_test, pred))
print("RMSE =", mean_squared_error(y_test, pred)**0.5)
print("R2   =", r2_score(y_test, pred))


In [ ]:
plt.figure(figsize=(5,5))
plt.scatter(y_test, pred)
lo = min(y_test.min(), pred.min())
hi = max(y_test.max(), pred.max())
plt.plot([lo,hi],[lo,hi],"--")
plt.xlabel("True density (g/cm³)")
plt.ylabel("Predicted density (g/cm³)")
plt.title("Real-CIF baseline")
plt.show()

importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
display(importance.to_frame("importance").head(10))


## 7. 从“能跑”升级到科研工作流

现在你已经完成了真正的：

`CIF → descriptor → target → merge → model`

下一步不能只是换一个更复杂的模型，而应依次增加：

1. **孔结构描述符**：LCD、PLD、ASA、void fraction、pore volume；
2. **化学环境描述符**：linkage、官能团、局部配位、RAC 等；
3. **可靠 target**：统一温度/压力/计算方法下的 CO₂ uptake / selectivity；
4. **更严格划分**：family/topology/structure-aware split；
5. **解释和筛选**：feature importance、SHAP、candidate ranking；
6. **最终回到结构**：检查 top candidates 的 CIF，而不是只看表格排名。

04C 将展示这一流程如何对应到真实 COF CO₂ 高通量筛选研究。


## Exercises

1. 把 `N_COF` 从 80 改为 150，记录解析失败率。
2. 加入 `cell_volume_A3` 和 `n_atoms`，观察模型性能为什么明显变化，并讨论是否构成 shortcut。
3. 画 `a/b/c`、density、element fraction 的分布图。
4. 找出密度最高和最低的 5 个 COF，并返回它们的 `COF_ID` 和名称。
5. 设计一个 `pore_features.csv`，写出你希望由 Zeo++ 计算的字段。
6. 用一句话解释为什么 `COF_ID` 是结构、描述符和 target 之间的“主键”。

### 本章完成标准

你能从一批真实 CIF 自动生成表格，并能清楚说明：
**哪些列来自 CIF、哪些列来自额外孔隙计算、target 从哪里来、以及它们如何通过 COF_ID 对齐。**


## 导出可追溯的数据表
一起保存描述符、target 和失败记录。连接另一数据库时需要已核验的 ID 对照，不能仅凭名称相似就认定为同一结构。


In [ ]:
from pathlib import Path
output_dir = Path('cof_cif_outputs')
output_dir.mkdir(exist_ok=True)
feature_table.to_csv(output_dir/'features.csv', index=False)
target_table.to_csv(output_dir/'targets.csv', index=False)
pd.DataFrame(failed, columns=['COF_ID','error']).to_csv(output_dir/'failed.csv', index=False)
print(output_dir.resolve())


## 数据来源与扩展阅读
[Dataset contracts / 数据使用约定](../docs/data_resources.md) · [COFSpace](https://github.com/gokhanonderaksu/COFSpace) · [CURATED-COFs](https://github.com/danieleongari/CURATED-COFs)
